# **OBJETIVOS DE ESTE CUADERNO**

Crear un corpus y sobre éste:


1.  Utilizar la librería SpaCy para:
  *  Tokenización (ok)
  *  Stop Words (ok Spacy)
  *  NER (Named Entities Recognition)  (ok)
  *  Lematización (ok Spacy)

  *  Bag Of Words
  *  TF-IDF
  *  POS-Tagging (Parts Of Speech)  (ok Spacy)


2.  Utilizar la librería NTLK para:
  *  Tokenización (ok)
  *  Stop Words (ok)
  *  NER (sólo en inglés) (ok)
  *  Stemming (ok)

3.   Utilizar Pandas para llevar los resultados de cada proceso a DataFrames




# **1. PREPARACIÓN DEL ENTORNO**

# **1.1   Instalación de librerías**

La siguiente celda tiene código para instalar la librería Spacy con el modelo "lg" (large) y asegurarse que todas las dependencias estén instaladas, así como instalar la librería NLTK

In [ ]:
import shutil
shutil.rmtree('/root/nltk_data', ignore_errors=True)

!pip install spacy
!python -m spacy download es_core_news_lg

!pip install nltk
!python -m nltk.downloader stopwords

# **1.2 Importar librerías y módulos**

In [ ]:
#########
# SPACY #
#########
import spacy

#displacy muestra análisis de forma más amigable
from spacy import displacy

########
# NLTK #
########
import nltk

#NLTK - Tokenización
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tokenize.regexp import blankline_tokenize

#NTLK - Lematización
from nltk.stem import WordNetLemmatizer

#NLTK - Stemming
from nltk.stem.snowball import SpanishStemmer

#NTLK - Stop Words
from nltk.corpus import stopwords

#NLTK - POS Tagging
from nltk import word_tokenize, pos_tag

#NTLK - NER
from nltk import ne_chunk


##################
# NUMPY Y PANDAS #
##################
import numpy as np
import pandas as pd

# **1.3 Descarga de modelos y componentes para la librería SpaCy**

In [ ]:
#En la siguiente celda, "es" significa "español" y "lg" significa "large"
nlp = spacy.load("es_core_news_lg")


# **1.4 Descarga de modelos y componentes para la librería NTLK**

In [ ]:
#Lematización
nltk.download('wordnet')
nltk.download('omw-1.4')

#Stop words
nltk.download('stopwords')

#Tokenización
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

#POS Tagging
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')

#NER
nltk.download('maxent_ne_chunker')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('words')
nltk.download('maxent_ne_chunker_tab')

# **1.5 Estructuras de datos auxiliares**

Las estructuras de esta celda se usan para convertir códigos arrojados por las librerías en un texto descriptivo.

Se explicarán a medida que se avance en el cuaderno y cada uno de los ejemplos

In [ ]:
tag_dict_nltk = {
    'CC': 'Conjunción coordinante',
    'CD': 'Número cardinal',
    'DT': 'Determinante',
    'EX': 'Allí existencial',
    'FW': 'Palabra extranjera',
    'IN': 'Preposición o conjunción subordinante',
    'JJ': 'Adjetivo',
    'JJR': 'Adjetivo comparativo',
    'JJS': 'Adjetivo superlativo',
    'LS': 'Marcador de elemento de lista',
    'MD': 'Modal',
    'NN': 'Sustantivo, singular o masa',
    'NNS': 'Sustantivo, plural',
    'NNP': 'Nombre propio, singular',
    'NNPS': 'Nombre propio, plural',
    'PDT': 'Predeterminante',
    'POS': 'Terminación posesiva',
    'PRP': 'Pronombre personal',
    'PRP$': 'Pronombre posesivo',
    'RB': 'Adverbio',
    'RBR': 'Adverbio comparativo',
    'RBS': 'Adverbio superlativo',
    'RP': 'partícula',
    'SYM': 'Símbolo',
    'TO': 'hacia',
    'UH': 'interjección',
    'VB': 'Verbo, forma base',
    'VBD': 'Verbo, tiempo pasado',
    'VBG': 'Verbo, gerundio o participio presente',
    'VBN': 'Verbo, participio pasado',
    'VBP': 'Verbo, presente no 3ª persona del singular',
    'VBZ': 'Verbo, tercera persona del singular presente',
    'WDT': 'Wh-determinante',
    'WP': 'Pronombre Wh',
    'WP$': 'Pronombre posesivo',
    'WRB': 'Wh-adverbio'
}

# **2. Redacción del corpus y creación de objetos desde las librerías**

# **2.1 Definición del texto para aplicar análisis NLP**

In [ ]:
#NOTA: EL siguiente texto fue tomado de internet

texto = """Maria Salomea Skłodowska-Curie, conocida como Marie Curie, nació el 7 de noviembre de 1867 en Varsovia, Polonia. Fue una física, matemática y química pionera en el campo de la radiactividad. Marie no solamente fue la primera mujer, sino que la primera persona en recibir dos Premios Nobel en distintas especialidades: Física (1903) y Química (1911), y la primera mujer en hacer clases en la Universidad de París. Marie se casó con el científico Pierre Curie y fue madre de quien recibiría más adelante también un Premio Nobel en Química, Irène Joliot-Curie.

Motivada por el reciente descubrimiento de Antoine Henri Becquerel, que demostró que las sales de uranio emitían rayos de naturaleza desconocida, sin la necesidad de ser expuestas a la luz, Marie Curie investigó más allá este tema y descubrió que los compuestos formados por el torio también emitían rayos de forma espontánea. A este fenómeno le llamaron radioactividad.

Como la radiactividad que generaban algunas muestras era más fuerte de lo que se esperaba, Marie y Pierre sospecharon que había otra sustancia radiactiva más potente que el uranio y el torio, y en 1898 dieron a conocer públicamente que habían descubierto un nuevo elemento: el polonio. Unos meses después anunciaron su nuevo hallazgo: el radio.

En 1903 le concedieron el Premio Nobel de Física por el descubrimiento de los elementos radiactivos y en 1911 la otorgaron un segundo Nobel, el de Química, por sus investigaciones sobre el radio y sus compuestos.

Marie Curie fue nombrada directora del Instituto de Radio de París en 1914 y se fundó el Instituto Curie. Murió en 1934 producto de una anemia aplásica, la que se sospecha fue provocada por su alta exposición a la radiación.
"""

# **2.2 Objetos de cada librería que se usarán en el cuaderno**

In [ ]:
# La variable (o el objeto) "doc" contiene información después de aplicar NLP al texto
# "doc" será usada en todo el cuaderno

doc = nlp(texto)

# **3. TOKENIZACIÓN**

# **3.1 Tokenización básica como Serie**

Usando la librería NLTK, generar:

* Una variable llamada "words" que contenga los tokens, por palabra
* Una variable llamada "sentences" que contenga los tokens, por frase
* Una variable llamada "paragraphs" que contenta los tokens, por párrafo

Posteriormente crear un diccionario con estas 3 variables (esta parte ya está resuelta en la siguiente celda)

Luego, crear una Serie (Pandas) con nombre "serie1" que indique:

* Cantidad de párrafos
* Cantidad de frases
* Cantidad de palabras

---

Ejemplo de salida:

```
paragraphs      5
sentences      11
words         314
dtype: int64
```



In [ ]:
#Escribir aquí el código para tokenizar

words = word_tokenize(texto)
sentences = sent_tokenize(texto)
paragraphs = blankline_tokenize(texto)


#Creación del diccionario para posteriormente crear la serie
data = {'paragraphs': len(paragraphs), 'sentences': len(sentences), 'words': len(words)}


#Creación de la serie con Pandas
serie1 = pd.Series(data)



print(serie1)

# **4. TOKENIZACIÓN, STOP WORDS Y ATRIBUTOS**

# **4.1 Estadísticas básicas de un texto como Serie**

La variable "words2" en el siguiente código, contiene los tokens por palabras del texto en la sección 2.1.

Usando la librería SpaCy y la sintaxis de comprensión de Python, crear las siguientes variables a partir del contenido de "words2":

* all_stop_words: Lista (nativa) con los tokens que son stop words
* all_punctuation: Lista con tokens que son símbolos de puntuación
* all_digits: Lista con tokens que son dígitos
* all_alnum: Lista con tokens alfanuméricos

Posteriormente, crear una serie con nombre "serie1" que indique:

* Cantidad de stop words
* Cantidad de signos de puntuación
* Cantidad de dígitos
* Cantidad de palabras alfanuméricas

---

Ejemplo de salida:

```
stop words     145
punctuation     33
digits           9
alnum          270
dtype: int64
```

In [ ]:
# Tip de ayuda:
#
# La siguiente línea genera, por comprensión, una lista de números pares entre 1 y 9
# evens = [ x for x in range(10) if x % 2 == 0 ]
#

words2 = list(doc)

all_stop_words = [token for token in words2 if token.is_stop]
all_punctuation = [token for token in words2 if token.is_punct]
all_digits = [token for token in words2 if token.is_digit]
all_alnum = [token for token in words2 if token.text.isalnum()]

data = {
        'stop words': len(all_stop_words),
    'punctuation': len(all_punctuation),
    'digits': len(all_digits),
    'alnum': len(all_alnum)
}


serie1 = pd.Series(data)

print(serie1)

# **5. POS-TAGGING**

# **5.1 Pos-Tag palabra por palabra en un DataFrame**

Usando la librería NLTK, generar un DataFrame que contenga los tokens y el tag de cada token.

El índice del DataFrame es el token, y la columna es el tag


---

Ejemplo de salida:

```
                  tag
token                
Maria             NNP
Salomea           NNP
Skłodowska-Curie  NNP
,                   ,
conocida           NN
...               ...
exposición        VBP
a                  DT
la                 NN
radiación          NN
```

In [ ]:
# Tip de ayuda: usar la función 'pos_tag'

tokens = word_tokenize(texto)

#Escribir aquí el código
pos_tags = pos_tag(tokens)
df = pd.DataFrame(pos_tags, columns=['token', 'tag'])
df = df.set_index('token')


print(df)

# **5.2 Manipulación del DataFrame para adicionar el "named tag"**

Usando programación funcional, adicionar al DataFrame la columna "named_tag". Esta columna usará la estructura de datos de la sección 1.5 para indicar el nombre del tag.

---

Ejemplo de salida:

```
                  tag                                   named_tag
token                                                            
Maria             NNP                     Nombre propio, singular
Salomea           NNP                     Nombre propio, singular
Skłodowska-Curie  NNP                     Nombre propio, singular
,                   ,                                        None
conocida           NN                 Sustantivo, singular o masa
...               ...                                         ...
exposición        VBP  Verbo, presente no 3ª persona del singular
a                  DT                                Determinante
la                 NN                 Sustantivo, singular o masa
radiación          NN                 Sustantivo, singular o masa
.                   .                                        None

[314 rows x 2 columns]
```

In [ ]:
#Tip de ayuda: Usar la función "apply" de Pandas sobre el DataFrame creado en el ejercicio anterior

#Escribir aquí el código

df['named_tag'] = df['tag'].apply(lambda x: tag_dict_nltk.get(x, None))

print(df)

# **5.33 Agrupación (agregación) por tipo de tag**

Usando la función "groupby", crear el DataFrame "df" para indicar la cantidad de tags por tipo

---

Ejemplo de salida:

```
                                              tag
named_tag                                        
Adjetivo                                       22
Adverbio                                        3
Determinante                                    5
Nombre propio, singular                        47
Número cardinal                                 9
Palabra extranjera                             59
Preposición o conjunción subordinante          11
Sustantivo, plural                              6
Sustantivo, singular o masa                   108
Verbo, presente no 3ª persona del singular      5
Verbo, tercera persona del singular presente    1
Verbo, tiempo pasado                            5
```




In [ ]:
#Escribir aquí el código
df2 = df.groupby('named_tag').size().to_frame(name='tag')


print(df2)

# **6. LEMATIZACIÓN**

# **6.1 Lematización y DataFrame**

Usando la librería SpaCy, generar un DataFrame que tenga el valor de la palabra original como índice y una columna llamada "lema" con la palabra lematizada.

Para generar este DataFrame se requiere una estructura de datos nativa, la cual se sugiere que se genere en una sola línea usando comprensión

---

Ejemplo de salida:

```
                             lemma
Maria                        Maria
Salomea                    Salomea
Skłodowska-Curie  Skłodowska-Curie
,                                ,
conocida                  conocido
...                            ...
provocada                 provocar
alta                          alto
exposición              exposición
radiación                radiación
\n                              \n

[163 rows x 1 columns]
```



In [ ]:
lemma_df = pd.DataFrame(
    {token.text: token.lemma_ for token in doc}.items(),
    columns=['palabra', 'lema']
).set_index('palabra')

print(lemma_df)

# **6.2 Palabras más comunes después de lematizar**

Usando la función "groupby" y "sort_values", mostrar un resultado que indique las palabras que más aparecen en el texto (después de lematizar)

-----

Ejemplo de salida:

```
lemma
el         4
ser        4
él         3
uno        3
como       2
          ..
también    1
torio      1
tema       1
uranio     1
y          1
Length: 143, dtype: int64
```

In [ ]:
df2 = df.groupby('lemma').size().sort_values(ascending=False)
print(df2)